In [5]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import joblib
import time
import warnings
warnings.filterwarnings("ignore")

# Gabor features
def extract_gabor_features(gray):
    features = []
    num_orientations = 4
    for theta in range(num_orientations):
        theta_val = theta / num_orientations * np.pi
        kernel = cv2.getGaborKernel((7,7), 4.0, theta_val, 10.0, 0.5, 0, ktype=cv2.CV_32F)
        fimg = cv2.filter2D(gray, cv2.CV_8UC3, kernel)
        features.extend([np.mean(fimg), np.std(fimg)])
    return np.array(features)  # length 8

# Load dataset + color + gabor
def load_image_dataset(base_path, img_size=(128, 128)):
    X, y = [], []
    class_names = sorted([d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))])

    for label in tqdm(class_names, desc="Loading images"):
        class_dir = os.path.join(base_path, label)
        for img_file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_file)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue
                img = cv2.resize(img, img_size)

                # Convert color spaces (note: OpenCV default BGR)
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
                img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
                img_hsl = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HLS)
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

                # color means
                rgb_mean = np.mean(img_rgb.reshape(-1,3), axis=0)
                hsv_mean = np.mean(img_hsv.reshape(-1,3), axis=0)
                lab_mean = np.mean(img_lab.reshape(-1,3), axis=0)
                hsl_mean = np.mean(img_hsl.reshape(-1,3), axis=0)

                gabor_feats = extract_gabor_features(gray)

                feature_vector = np.concatenate([
                    rgb_mean,    # 3
                    hsv_mean,    # 3
                    hsl_mean,    # 3
                    lab_mean,    # 3
                    gabor_feats  # 8
                ])

                X.append(feature_vector)
                y.append(label)
            except Exception as e:
                print("Error membaca:", img_path, e)
                continue

    return np.array(X), np.array(y)

# Pipeline
def make_pipeline(mode="none", X_train=None, y_train=None, n_pca=None, n_lda=None):
    """Membuat pipeline berdasarkan mode reduksi fitur"""
    steps = [('scaler', StandardScaler())]

    # Validasi dimensi
    if X_train is not None:
        n_features = X_train.shape[1]
        n_classes = len(np.unique(y_train))
        
        if mode == "pca" and n_pca is not None:
            n_pca = min(n_pca, n_features)
            steps.append(('pca', PCA(n_components=n_pca, random_state=42)))
            
        elif mode == "lda" and n_lda is not None:
            n_lda = max(1, min(n_lda, n_classes - 1))
            steps.append(('lda', LDA(n_components=n_lda)))
            
        elif mode == "pca+lda" and n_pca is not None and n_lda is not None:
            n_pca = min(n_pca, n_features)
            n_lda = max(1, min(n_lda, n_classes - 1))
            steps.append(('pca', PCA(n_components=n_pca, random_state=42)))
            steps.append(('lda', LDA(n_components=n_lda)))

    steps.append(('svm', SVC(probability=True, class_weight='balanced')))
    return Pipeline(steps)

# Top-2 Accuracy
def top2_accuracy(model, X, y_true):
    prob = model.predict_proba(X)
    top2 = np.argsort(prob, axis=1)[:, -2:]
    classes = model.classes_
    top2_labels = classes[top2]
    correct = sum([y_true[i] in top2_labels[i] for i in range(len(y_true))])
    return correct / len(y_true)

# Main training flow - MULTI MODE
if __name__ == "__main__":
    base_dataset = "./thread_data"
    img_size = (128, 128)

    print("=== LOADING DATASET ===")
    X, y = load_image_dataset(base_dataset, img_size=img_size)
    print("Shape fitur:", X.shape)
    print("Jumlah data:", len(y))
    print("Jumlah kelas:", len(np.unique(y)))

    if len(y) == 0:
        raise RuntimeError("Dataset kosong atau path salah.")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # Parameter grid untuk GridSearchCV
    param_grid = {
        'svm__C': [0.5, 1, 10, 50],
        'svm__kernel': ['rbf'],
        'svm__gamma': [0.5, 1, 10, 50]
    }

    # Semua mode yang akan diuji
    modes = ["none", "pca", "lda", "pca+lda"]
    results = []

    print("\n=== TRAINING SEMUA MODE REDUKSI FITUR ===")
    os.makedirs("./svm_results", exist_ok=True)

    for mode in modes:
        print(f"\n--- MODE: {mode.upper()} ---")
        
        # Tentukan hyperparameter reduksi untuk mode ini
        if mode == "pca":
            n_pca, n_lda = 10, None  # PCA saja
        elif mode == "lda":
            n_pca, n_lda = None, 5   # LDA saja (max n_classes-1)
        elif mode == "pca+lda":
            n_pca, n_lda = 10, 5     # PCA + LDA
        else:  # none
            n_pca, n_lda = None, None

        # Buat pipeline
        start_time = time.time()
        pipeline = make_pipeline(mode, X_train, y_train, n_pca=n_pca, n_lda=n_lda)
        
        # GridSearchCV
        grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)
        grid.fit(X_train, y_train)
        train_time = time.time() - start_time

        # Test time measurement
        start_test = time.time()
        y_pred = grid.predict(X_test)
        top2_acc = top2_accuracy(grid.best_estimator_, X_test, y_test)
        test_time = time.time() - start_test
        
        # Metrics
        acc = accuracy_score(y_test, y_pred)
        total_time = train_time + test_time

        print(f"Akurasi Top-1: {acc:.4f}")
        print(f"Akurasi Top-2: {top2_acc:.4f}")
        print(f"Waktu Training: {train_time:.2f}s")
        print(f"Waktu Testing:  {test_time:.2f}s")
        print(f"Total Waktu:    {total_time:.2f}s")
        print("Best Params:", grid.best_params_)

        # Simpan model terbaik untuk mode ini
        model_filename = f"./svm_results/pipeline_model_{mode}.pkl"
        joblib.dump(grid.best_estimator_, model_filename)
        print(f"Model disimpan: {model_filename}")

        # Simpan hasil ke list
        results.append({
            'mode': mode,
            'top1_acc': acc,
            'top2_acc': top2_acc,
            'train_time': train_time,
            'test_time': test_time,
            'total_time': total_time,
            'best_C': grid.best_params_['svm__C'],
            'best_gamma': grid.best_params_['svm__gamma'],
            'model_path': model_filename
        })

    # SUMMARY & BEST MODEL
    df_results = pd.DataFrame(results)
    df_results.to_csv("./svm_results/all_modes_comparison.csv", index=False)
    
    print("\n=== RINGKASAN SEMUA MODE ===")
    print(df_results[['mode', 'top1_acc', 'top2_acc', 'total_time', 'best_C', 'best_gamma']].round(4))
    
    # Cari model terbaik berdasarkan Top-1 Accuracy
    best_model_idx = df_results['top1_acc'].idxmax()
    best_model = df_results.iloc[best_model_idx]
    
    print(f"\nMODEL TERBAIK (Top-1 Acc):")
    print(f"  Mode: {best_model['mode']}")
    print(f"  Akurasi Top-1: {best_model['top1_acc']:.4f}")
    print(f"  Total Waktu: {best_model['total_time']:.2f}s")
    print(f"  Model path: {best_model['model_path']}")
    
    # Simpan link ke best model
    df_results.to_csv("./svm_results/results_summary.csv", index=False)
    print("\nSemua hasil lengkap tersimpan di:")
    print("- all_modes_comparison.csv (perbandingan semua mode)")
    print("- results_summary.csv (kompatibilitas lama)")
    print("- pipeline_model_[mode].pkl (masing-masing model)")

=== LOADING DATASET ===


Loading images: 100%|████████████████████████████████████████████████████████████████| 200/200 [08:56<00:00,  2.68s/it]


Shape fitur: (10000, 20)
Jumlah data: 10000
Jumlah kelas: 200

=== TRAINING SEMUA MODE REDUKSI FITUR ===

--- MODE: NONE ---
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Akurasi Top-1: 0.9465
Akurasi Top-2: 0.9930
Waktu Training: 239.77s
Waktu Testing:  8.63s
Total Waktu:    248.40s
Best Params: {'svm__C': 10, 'svm__gamma': 0.5, 'svm__kernel': 'rbf'}
Model disimpan: ./svm_results/pipeline_model_none.pkl

--- MODE: PCA ---
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Akurasi Top-1: 0.9460
Akurasi Top-2: 0.9930
Waktu Training: 205.03s
Waktu Testing:  8.37s
Total Waktu:    213.41s
Best Params: {'svm__C': 10, 'svm__gamma': 0.5, 'svm__kernel': 'rbf'}
Model disimpan: ./svm_results/pipeline_model_pca.pkl

--- MODE: LDA ---
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Akurasi Top-1: 0.9065
Akurasi Top-2: 0.9525
Waktu Training: 29478.62s
Waktu Testing:  12.78s
Total Waktu:    29491.40s
Best Params: {'svm__C': 1, 'svm__gamma': 0.5, 'svm__kernel': '

KHUSUS YANG TERBAIK SETIAP SKENARIO REDUKSI BERDASARKAN HASIL SEBELUMNYA

In [7]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import joblib
import time
import warnings
warnings.filterwarnings("ignore")

# Gabor features
def extract_gabor_features(gray):
    features = []
    num_orientations = 4
    for theta in range(num_orientations):
        theta_val = theta / num_orientations * np.pi
        kernel = cv2.getGaborKernel((7,7), 4.0, theta_val, 10.0, 0.5, 0, ktype=cv2.CV_32F)
        fimg = cv2.filter2D(gray, cv2.CV_8UC3, kernel)
        features.extend([np.mean(fimg), np.std(fimg)])
    return np.array(features)

# Load dataset
def load_image_dataset(base_path, img_size=(128, 128)):
    X, y = [], []
    class_names = sorted([d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))])

    for label in tqdm(class_names, desc="Loading images"):
        class_dir = os.path.join(base_path, label)
        for img_file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_file)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue
                img = cv2.resize(img, img_size)

                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
                img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
                img_hsl = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HLS)
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

                rgb_mean = np.mean(img_rgb.reshape(-1,3), axis=0)
                hsv_mean = np.mean(img_hsv.reshape(-1,3), axis=0)
                lab_mean = np.mean(img_lab.reshape(-1,3), axis=0)
                hsl_mean = np.mean(img_hsl.reshape(-1,3), axis=0)

                gabor_feats = extract_gabor_features(gray)

                feature_vector = np.concatenate([
                    rgb_mean, hsv_mean, hsl_mean, lab_mean, gabor_feats
                ])

                X.append(feature_vector)
                y.append(label)
            except Exception:
                continue

    return np.array(X), np.array(y)

# Top-2 Accuracy
def top2_accuracy(model, X, y_true):
    prob = model.predict_proba(X)
    top2 = np.argsort(prob, axis=1)[:, -2:]
    classes = model.classes_
    top2_labels = classes[top2]
    correct = sum([y_true[i] in top2_labels[i] for i in range(len(y_true))])
    return correct / len(y_true)

# 4) PIPELINE
def create_pipeline(mode="none", C=1, gamma=0.5, n_pca=10, n_lda=5, X_train=None, y_train=None):
    steps = [('scaler', StandardScaler())]
    
    n_features = X_train.shape[1] if X_train is not None else 20
    n_classes = len(np.unique(y_train)) if y_train is not None else 10
    
    if mode == "pca":
        n_pca = min(n_pca, n_features)
        steps.append(('pca', PCA(n_components=n_pca, random_state=42)))
    elif mode == "lda":
        n_lda = min(n_lda, n_classes - 1)
        steps.append(('lda', LDA(n_components=n_lda)))
    elif mode == "pca+lda":
        n_pca = min(n_pca, n_features)
        n_lda = min(n_lda, n_classes - 1)
        steps.append(('pca', PCA(n_components=n_pca, random_state=42)))
        steps.append(('lda', LDA(n_components=n_lda)))
    
    # SVM dengan params spesifik
    steps.append(('svm', SVC(C=C, gamma=gamma, kernel='rbf', probability=True, class_weight='balanced')))
    return Pipeline(steps)

# MAIN
if __name__ == "__main__":
    base_dataset = "./thread_data"
    img_size = (128, 128)

    print("LOADING DATASET")
    X, y = load_image_dataset(base_dataset, img_size=img_size)
    print(f"Dataset: {X.shape} | Kelas: {len(np.unique(y))}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # BEST PARAMS PER MODE
    modes_config = {
        "none":     {"C": 10, "gamma": 0.5},
        "pca":      {"C": 10, "gamma": 0.5},
        "lda":      {"C": 1,  "gamma": 0.5},
        "pca+lda":  {"C": 1,  "gamma": 0.5}
    }
    
    results = []
    os.makedirs("./svm_results_fast", exist_ok=True)

    print("\nTRAINING 4 MODE")

    for mode_name, params in modes_config.items():
        print(f"\n--- {mode_name.upper()} (C={params['C']}, γ={params['gamma']}) ---")
        
        start_total = time.time()
        
        # Pass X_train, y_train ke pipeline factory
        pipeline = create_pipeline(
            mode=mode_name,
            C=params['C'],
            gamma=params['gamma'],
            X_train=X_train,
            y_train=y_train
        )
        
        # TRAINING
        start_train = time.time()
        pipeline.fit(X_train, y_train)  # Sekarang OK!
        train_time = time.time() - start_train
        
        # TESTING
        start_test = time.time()
        y_pred = pipeline.predict(X_test)
        top2_acc = top2_accuracy(pipeline, X_test, y_test)
        test_time = time.time() - start_test
        
        total_time = time.time() - start_total
        acc = accuracy_score(y_test, y_pred)
        
        # Jumlah fitur akhir
        try:
            # Fit transform untuk cek dimensi
            X_train_transformed = pipeline[:-1].fit_transform(X_train, y_train)
            n_final_features = X_train_transformed.shape[1]
        except:
            n_final_features = X_train.shape[1]

        print(f"   Top-1 Acc:   {acc:.4f}")
        print(f"   Top-2 Acc:   {top2_acc:.4f}")
        print(f"   Fitur Akhir: {n_final_features}")
        print(f"   Train Time:  {train_time:.2f}s")
        print(f"   Test Time:   {test_time:.2f}s")
        print(f"   TOTAL:       {total_time:.2f}s")

        # Simpan Model
        model_name = f"model_{mode_name}_C{params['C']}_g{params['gamma']}"
        model_path = f"./svm_results_fast/{model_name}.pkl"
        joblib.dump(pipeline, model_path)

        results.append({
            'mode': mode_name,
            'C': params['C'],
            'gamma': params['gamma'],
            'top1_acc': acc,
            'top2_acc': top2_acc,
            'n_features': n_final_features,
            'train_time': train_time,
            'test_time': test_time,
            'total_time': total_time,
            'model_path': model_path
        })

    # Tabel Komparasi
    df_results = pd.DataFrame(results)
    df_results.to_csv("./svm_results_fast/comparison_table.csv", index=False)
    
    print("\n" + "="*70)
    print("TABEL HASIL UNTUK SKRIPSI")
    print("="*70)
    print(df_results[['mode', 'C', 'gamma', 'top1_acc', 'top2_acc', 'n_features', 'total_time']].round(4))
    
    # BEST MODE
    best_acc_idx = df_results['top1_acc'].idxmax()
    best_speed_idx = df_results['total_time'].idxmin()
    
    best_acc = df_results.iloc[best_acc_idx]
    best_speed = df_results.iloc[best_speed_idx]
    
    print(f"\nTERBAIK AKURASI: {best_acc['mode']} | {best_acc['top1_acc']:.4f} | {best_acc['model_path']}")
    print(f"TERCEPAT:       {best_speed['mode']} | {best_speed['total_time']:.2f}s | {best_speed['model_path']}")
    
    print(f"\nFolder hasil: ./svm_results_fast/")
    print(f"Copy table ke skripsi: comparison_table.csv")

LOADING DATASET


Loading images: 100%|████████████████████████████████████████████████████████████████| 200/200 [10:18<00:00,  3.09s/it]


Dataset: (10000, 20) | Kelas: 200

TRAINING 4 MODE

--- NONE (C=10, γ=0.5) ---
   Top-1 Acc:   0.9465
   Top-2 Acc:   0.9935
   Fitur Akhir: 20
   Train Time:  10.95s
   Test Time:   9.43s
   TOTAL:       20.38s

--- PCA (C=10, γ=0.5) ---
   Top-1 Acc:   0.9460
   Top-2 Acc:   0.9930
   Fitur Akhir: 10
   Train Time:  9.60s
   Test Time:   9.49s
   TOTAL:       19.09s

--- LDA (C=1, γ=0.5) ---
   Top-1 Acc:   0.9065
   Top-2 Acc:   0.9550
   Fitur Akhir: 5
   Train Time:  37.68s
   Test Time:   11.66s
   TOTAL:       49.35s

--- PCA+LDA (C=1, γ=0.5) ---
   Top-1 Acc:   0.9150
   Top-2 Acc:   0.9520
   Fitur Akhir: 5
   Train Time:  36.88s
   Test Time:   12.10s
   TOTAL:       48.97s

TABEL HASIL UNTUK SKRIPSI
      mode   C  gamma  top1_acc  top2_acc  n_features  total_time
0     none  10    0.5    0.9465    0.9935          20     20.3807
1      pca  10    0.5    0.9460    0.9930          10     19.0935
2      lda   1    0.5    0.9065    0.9550           5     49.3488
3  pca+lda   1  